In [ ]:

# PART 1: TRAIN WITH VISDRONE

!pip -q install ultralytics requests pillow
import os
import shutil
import zipfile
import requests
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from google.colab import files
import torch, gc


# Paths

# Creates a new directory to store both raw and processed dataset
ROOT = Path("/content/VisDroneVehicle")
OUT = ROOT / "vehicle_yolo"
ROOT.mkdir(parents=True, exist_ok=True)


# Download helper

# Helps us download the file in chucks instead of downloading everything all at once.
# Also helps us find out if the donwload was done corredctly or not.
def download_file(url, save_path):
    print(f"Downloading: {url}")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(save_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)


# Download VisDrone DET train + val and extracts them into folders

downloads = {
    "train": {
        "url": "https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-train.zip",
        "zip_path": ROOT / "VisDrone2019-DET-train.zip",
        "extract_path": ROOT / "raw_train",
    },
    "val": {
        "url": "https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-val.zip",
        "zip_path": ROOT / "VisDrone2019-DET-val.zip",
        "extract_path": ROOT / "raw_val",
    }
}

# Sees if we already have the vidrone downloaded or not. If so then skip donwloading

for split, info in downloads.items():
    if not info["zip_path"].exists():
        download_file(info["url"], info["zip_path"])

# It checks if it has been extracted from zip.
    if not info["extract_path"].exists():
        print(f"Extracting {split}...")
        with zipfile.ZipFile(info["zip_path"], "r") as zf:
            zf.extractall(info["extract_path"])

print("Download/extract complete.")


# Build YOLO-format dataset
# Keep only vehicle-like classes
# VisDrone: 4 car, 5 van, 6 truck, 9 bus, 10 motor
# Map all to class 0 = vehicle

# Helps us delete old processed datasets if we already have them
if OUT.exists():
    shutil.rmtree(OUT)

# Creates a YOLO folder strcuture which is need in YOLO.
for split in ["train", "val"]:
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

# Keeps class of cars, bus, and etc.
KEEP_CLASSES = {4, 5, 6, 9, 10}

# Helps us find the visdrone downloaded folder.
def find_visdrone_folder(base: Path, split: str) -> Path:
    candidates = list(base.rglob(f"VisDrone2019-DET-{split}"))
    if not candidates:
        raise FileNotFoundError(f"Could not find extracted VisDrone folder for split={split}")
    return candidates[0]


# Helps find the visdrone folder and points to both images and labels and tells us wehre they would go.
def convert_split(split: str):
    src_root = find_visdrone_folder(ROOT / f"raw_{split}", split)
    img_dir = src_root / "images"
    ann_dir = src_root / "annotations"

    out_img = OUT / "images" / split
    out_lbl = OUT / "labels" / split

# Grabs all the images in the datasets
    image_files = sorted(img_dir.glob("*.jpg"))
    print(f"{split}: found {len(image_files)} images")

    kept_boxes = 0

# Find each image with coressponding label file.
    for img_path in image_files:
        ann_path = ann_dir / f"{img_path.stem}.txt"
        if not ann_path.exists():
            continue

        w, h = Image.open(img_path).size
        yolo_lines = []

        with open(ann_path, "r", encoding="utf-8") as f:
            for line in f:

  # Reads Visdrone format such as x, y, height and base.

                parts = line.strip().split(",")
                if len(parts) < 8:
                    continue

                x, y, bw, bh = map(float, parts[:4])
                score = int(parts[4])
                cls_id = int(parts[5])

                if score == 0:
                    continue
                if cls_id not in KEEP_CLASSES:
                    continue
    # Helps us convert the Visdrone Format into YOLO format with x_center, y_center and etc.
                xc = (x + bw / 2.0) / w
                yc = (y + bh / 2.0) / h
                nw = bw / w
                nh = bh / h

    #Converts the vehicles into one class.

                yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
                kept_boxes += 1

    # Yolo format labels and  move into YOLO folders

        shutil.copy2(img_path, out_img / img_path.name)
        with open(out_lbl / f"{img_path.stem}.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(yolo_lines))

    print(f"{split}: kept {kept_boxes} vehicle boxes")

convert_split("train")
convert_split("val")


# creates a YAML file that tells YOLO where the trainning images and validation is located and defines the class labels.

yaml_text = """
path: /content/VisDroneVehicle/vehicle_yolo
train: images/train
val: images/val

names:
  0: vehicle
"""

with open("/content/visdrone_vehicle.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_text)

print(yaml_text)


# Train lightweight model

gc.collect()
torch.cuda.empty_cache()

model = YOLO("yolo11n.pt")

model.train(
    data="/content/visdrone_vehicle.yaml",
    epochs=3,
    imgsz=640,
    batch=4,
    workers=1,
    pretrained=True,
    project="/content/runs",
    name="visdrone_vehicle_fast"
)


# Download best model

best_model_path = "/content/runs/visdrone_vehicle_fast/weights/best.pt"
print("Downloading trained model:", best_model_path)
files.download(best_model_path)

Downloading: https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-train.zip
Extracting train...
Downloading: https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-val.zip
Extracting val...
Download/extract complete.
train: found 6471 images
train: kept 218271 vehicle boxes
val: found 548 images
val: kept 21926 vehicle boxes

path: /content/VisDroneVehicle/vehicle_yolo
train: images/train
val: images/val

names:
  0: vehicle

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/visdrone_vehicle.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2en

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip -q install filterpy scikit-image

import os
import sys
import shutil

# Re-clone SORT cleanly
if os.path.exists("/content/sort"):
    shutil.rmtree("/content/sort")

!git clone https://github.com/abewley/sort.git /content/sort

# Fix the bad TkAgg line inside sort.py
sort_file = "/content/sort/sort.py"

with open(sort_file, "r") as f:
    code = f.read()

code = code.replace(
    "matplotlib.use('TkAgg')",
    "matplotlib.use('Agg')"
)

code = code.replace(
    'matplotlib.use("TkAgg")',
    'matplotlib.use("Agg")'
)

with open(sort_file, "w") as f:
    f.write(code)

# Now import SORT
sys.path.insert(0, "/content/sort")
from sort import Sort

print("SORT fixed and imported successfully.")

Cloning into '/content/sort'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 208 (delta 45), reused 40 (delta 40), pack-reused 159 (from 1)
Receiving objects: 100% (208/208), 1.20 MiB | 21.62 MiB/s, done.
Resolving deltas: 100% (76/76), done.
SORT fixed and imported successfully.


In [ ]:


!pip -q install ultralytics requests pillow
!pip -q install ultralytics filterpy lap

# IMPORTS

import csv
import os
import sys
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from ultralytics import YOLO
from google.colab import files


# SIMPLE CSV LISTS

traffic_behavior_rows = []
vehicle_size_rows = []


#THRESHOLD SETTING

# Use 0.0375 if you want about 72 pixels for a 1920-width video:
# 1920 * 0.0375 = 72 px

SIZE_THRESHOLD_SCALE = 0.0375


# CLONE SORT

if not os.path.exists("/content/sort"):
    !git clone https://github.com/abewley/sort.git

sys.path.append("/content/sort")
from sort import Sort


# LOAD MODEL

print("Upload your trained best.pt file")

uploaded_model = files.upload()
model_path = list(uploaded_model.keys())[0]

model = YOLO(model_path)


# LOAD VIDEO

print("Upload your traffic video file")

uploaded_video = files.upload()
video_path = list(uploaded_video.keys())[0]

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 30.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Frame size:", width, "x", height)
print("Total frames:", total_frames)

print("Vehicle size threshold scale:", SIZE_THRESHOLD_SCALE)
print("Vehicle size threshold px:", width * SIZE_THRESHOLD_SCALE)


# OUTPUT VIDEO

output_path = "early_late_merge_identifier_with_csvs_histogram_and_mph.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)


# MPH SETTINGS

# Calibrated lower because 0.75 ft/pixel was making speeds way too high
FEET_PER_PIXEL = 0.28

# Longer window makes speed less sensitive to bounding box shaking
SPEED_WINDOW = 35

# Ignore tiny movements caused by detection jitter
MIN_PIXELS_MOVED = 2.5

# Keep display realistic
MAX_REASONABLE_MPH = 80

# Smooth speed changes more heavily
MPH_SMOOTHING = 0.92


# VEHICLE CLASSES

vehicle_names = [
    "vehicle",
    "car",
    "truck",
    "bus",
    "motorcycle",
    "van"
]


# TRACKERS

trackers = [
    Sort(max_age=25, min_hits=1, iou_threshold=0.1)
]

prev_boxes = {0: {}}


# MERGE GEOMETRY

exit_x = 85
bottom_y = 265


# TRACK STATES

track_states = {0: {}}

merge_count = 0

counted_merge_ids = set()

recent_counted_centers = []

count_distance_thresh = 110

count_memory_frames = int(3.0 * fps)

min_merge_frames_to_count = 20

guide_alpha = 0.38

guide_thickness = 2


# HIGHWAY REGIONS

# This keeps the detector focused only on the upper freeway where the merge happens.
highway_regions = [
    (70, 275)
]

# One YOLO confidence value for the one highway region above.
region_confs = [
    0.30
]

# Normal non-merging vehicle color.
colors = [
    (255, 0, 0)
]

# Original merge color kept for compatibility with the older merge logic.
merge_color = (0, 255, 255)


# EARLY VS LATE MERGE GEOMETRY

# Overall red merge corridor.
# Any tracked vehicle touching this corridor is treated as generally MERGING.
# The bottom edge is moved UP so middle-lane cars are not counted as late merging.
merge_polygon = np.array([
    [75, 148],
    [width - 55, 148],
    [width - 55, 240],
    [75, 240]
], dtype=np.int32)


# Entry polygon follows the same corridor closely so the original merge logic still works.
entry_polygon = np.array([
    [75, 160],
    [width - 55, 160],
    [width - 55, 252],
    [75, 252]
], dtype=np.int32)

# Bright green early-decision region on the right side of the merge.
# This is only the top merge lane, not the middle freeway lane.
early_region_polygon = np.array([
    [1245, 166],
    [width - 70, 166],
    [width - 70, 248],
    [1245, 248]
], dtype=np.int32)

# Yellow late-decision region on the left side of the blue line.
# Bottom edge moved up so yellow late boxes do not include middle-lane traffic.
late_region_polygon = np.array([
    [75, 154],
    [1245, 154],
    [1245, 236],
    [75, 236]
], dtype=np.int32)

# Blue divider line between early region and late region.
# If a vehicle reaches the left side of this line without crossing the dark green line,
# it becomes a permanent LATE MERGE.
late_blue_line = np.array([
    [1245, 135],
    [1245, 270]
], dtype=np.int32)


# Dark green lane-crossing line.
# This line is intentionally shorter and only covers the actual early-merge lane-change zone.
# A vehicle must cross this line from ABOVE to BELOW to count as early.
dark_green_line = np.array([
    [1285, 209],
    [width, 209]
], dtype=np.int32)

early_merge_count = 0
late_merge_count = 0

early_merge_ids = set()
late_merge_ids = set()

early_color = (0, 255, 0)
late_color = (0, 255, 255)
currently_merging_color = (120, 255, 120)


# VEHICLE SIZE CLASSIFIER

def classify_vehicle_size(best_label, x1_full, y1_full, x2_full, y2_full, width):

    box_width = x2_full - x1_full
    box_height = y2_full - y1_full

    raw_vehicle_size = max(box_width, box_height)

    aspect_ratio = box_width / box_height if box_height > 0 else 0

    size_threshold = width * SIZE_THRESHOLD_SCALE

    # YOLO label rule:
    # If YOLO identifies it as a truck, bus, or van, classify it as big.
    if best_label in ["truck", "bus", "van"]:
        return "big car"

    # Long vehicle rule:
    # Helps classify long trailers/semi-trucks that may not be very tall.
    elif aspect_ratio > 2.2 and box_width > width * 0.045:
        return "big car"

    # Pixel-size rule:
    # If the max bounding-box dimension is larger than the threshold, classify as big.
    elif raw_vehicle_size > size_threshold:
        return "big car"

    else:
        return "small car"


# IOU
def compute_iou(boxA, boxB):

    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])

    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)

    inter_area = inter_w * inter_h

    boxA_area = max(0, boxA[2]-boxA[0]) * max(0, boxA[3]-boxA[1])
    boxB_area = max(0, boxB[2]-boxB[0]) * max(0, boxB[3]-boxB[1])

    union = boxA_area + boxB_area - inter_area

    if union == 0:
        return 0.0

    return inter_area / union


# POLYGON HELPERS

def point_in_polygon(point, polygon):
    return cv2.pointPolygonTest(polygon, point, False) >= 0

def box_touches_polygon(x1, y1, x2, y2, polygon):

    test_points = [
        (int(x1), int(y2)),
        (int((x1 + x2) / 2), int(y2)),
        (int(x2), int(y2)),
        (int((x1 + x2) / 2), int((y1 + y2) / 2))
    ]

    for pt in test_points:
        if point_in_polygon(pt, polygon):
            return True

    return False

def bottom_boundary_y_at_x(x, polygon):

    x_left, y_left = polygon[3]
    x_right, y_right = polygon[2]

    if x_right == x_left:
        return y_left

    t = (x - x_left) / (x_right - x_left)

    return y_left + t * (y_right - y_left)

def line_y_at_x(x, line):

    x1, y1 = line[0]
    x2, y2 = line[1]

    if x2 == x1:
        return y1

    t = (x - x1) / (x2 - x1)

    return y1 + t * (y2 - y1)

def point_is_left_of_vertical_line(point, line):

    px, py = point

    line_x = int(line[0][0])

    return px <= line_x



# UPDATE CSV ROW MERGE TYPE

def update_saved_merge_type(traffic_behavior_rows, track_id, merge_type):

    if merge_type not in ["early", "late"]:
        return

    for row in traffic_behavior_rows:

        if row["Track ID"] == track_id:

            row["Merge Type"] = (
                "EARLY"
                if merge_type == "early"
                else "LATE"
            )


# PROCESS VIDEO

frame_count = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    annotated_full = frame.copy()

    for region_idx, (y_start, y_end) in enumerate(highway_regions):

        road_frame = frame[y_start:y_end, :]

        if road_frame.shape[0] == 0:
            continue

        conf_val = region_confs[region_idx]

        results = model(
            road_frame,
            conf=conf_val,
            imgsz=1536,
            verbose=False
        )[0]

        detections = []
        det_meta = []

        if results.boxes is not None:

            for box in results.boxes:

                cls_id = int(box.cls[0])
                conf = float(box.conf[0])

                label = model.names[cls_id]

                if label not in vehicle_names:
                    continue

                x1, y1, x2, y2 = map(float, box.xyxy[0])

                if (x2 - x1) < 18 or (y2 - y1) < 12:
                    continue

                detections.append([x1, y1, x2, y2, conf])

                det_meta.append({
                    "box": [x1, y1, x2, y2],
                    "label": label,
                    "conf": conf
                })

        detections = np.array(detections) if len(detections) > 0 else np.empty((0, 5))

        tracked_objects = trackers[region_idx].update(detections)

        for track in tracked_objects:

            x1, y1, x2, y2, track_id = track.astype(int)

            best_iou = 0
            best_label = "vehicle"

            for d in det_meta:

                iou = compute_iou([x1, y1, x2, y2], d["box"])

                if iou > best_iou:
                    best_iou = iou
                    best_label = d["label"]

            if track_id in prev_boxes[region_idx]:

                px1, py1, px2, py2 = prev_boxes[region_idx][track_id]

                alpha = 0.55

                x1 = int(alpha * px1 + (1 - alpha) * x1)
                y1 = int(alpha * py1 + (1 - alpha) * y1)
                x2 = int(alpha * px2 + (1 - alpha) * x2)
                y2 = int(alpha * py2 + (1 - alpha) * y2)

            prev_boxes[region_idx][track_id] = (x1, y1, x2, y2)

            y1_full = y1 + y_start
            y2_full = y2 + y_start

            x1_full = x1
            x2_full = x2

            if track_id not in track_states[region_idx]:

                track_states[region_idx][track_id] = {

                    "is_merging": False,
                    "merge_done": False,
                    "last_seen_frame": frame_count,
                    "merge_frame_count": 0,

                    "merge_type": None,
                    "has_entered_merge_corridor": False,
                    "has_been_above_dark_green_line": False,
                    "early_merge_latched": False,
                    "late_merge_latched": False,
                    "was_in_early_region": False,
                    "has_crossed_blue_line": False,

                    "vehicle_label": best_label,
                    "vehicle_size": "small car",

                    "entry_frame": None,
                    "entry_time": None,

                    "prev_x": None,
                    "prev_y": None,
                    "prev_frame": None,
                    "prev_center_y": None,
                    "prev_center_x": None,
                    "early_line_frame_count": 0,

                    "x_history": [],
                    "speed_mph": 0.0,

                    "speed_values": [],
                    "mph_values": [],
                    "size_vote_history": [],
                    "stable_vehicle_size": "small car"
                }

            state = track_states[region_idx][track_id]

            state["last_seen_frame"] = frame_count

            # Permanent label sync:
            # Once a vehicle has crossed the dark green line, this keeps it EARLY forever.
            if state.get("early_merge_latched", False):

                state["merge_type"] = "early"

                if track_id not in early_merge_ids:
                    early_merge_ids.add(track_id)

                early_merge_count = len(early_merge_ids)

            # Same idea for late merges.
            if state.get("late_merge_latched", False):

                state["merge_type"] = "late"

                if track_id not in late_merge_ids:
                    late_merge_ids.add(track_id)

                late_merge_count = len(late_merge_ids)

            bottom_center_x = int((x1_full + x2_full) / 2)
            bottom_center_y = int(y2_full)



            # EARLY VS LATE MERGE CLASSIFICATION

            center_x = int((x1_full + x2_full) / 2)
            center_y = int((y1_full + y2_full) / 2)

            center_point = (center_x, center_y)
            bottom_point = (bottom_center_x, bottom_center_y)

            touches_red_corridor = box_touches_polygon(
                x1_full,
                y1_full,
                x2_full,
                y2_full,
                merge_polygon
            )

            center_inside_early_region = point_in_polygon(
                center_point,
                early_region_polygon
            )

            bottom_inside_early_region = point_in_polygon(
                bottom_point,
                early_region_polygon
            )

            center_inside_late_region = point_in_polygon(
                center_point,
                late_region_polygon
            )

            bottom_inside_late_region = point_in_polygon(
                bottom_point,
                late_region_polygon
            )

            if touches_red_corridor:
                state["has_entered_merge_corridor"] = True

            dark_line_x_min = min(dark_green_line[0][0], dark_green_line[1][0])
            dark_line_x_max = max(dark_green_line[0][0], dark_green_line[1][0])


            # EARLY MERGE:
            # Vehicle must move from above the dark green line to below it.

            dark_line_y_now = line_y_at_x(center_x, dark_green_line)

            box_overlaps_dark_line_x = (
                x2_full >= dark_line_x_min
                and x1_full <= dark_line_x_max
            )

            crossed_dark_green_line = (
                state["has_entered_merge_corridor"]
                and center_inside_early_region
                and box_overlaps_dark_line_x
                and state["prev_center_y"] is not None
                and state["prev_center_y"] < dark_line_y_now
                and center_y >= dark_line_y_now
            )

            if (
              crossed_dark_green_line
              and not state.get("early_merge_latched", False)
              and not state.get("late_merge_latched", False)
          ):

              state["merge_type"] = "early"
              state["early_merge_latched"] = True

              early_merge_ids.add(track_id)
              early_merge_count = len(early_merge_ids)

              update_saved_merge_type(
                  traffic_behavior_rows,
                  track_id,
                  state["merge_type"]
              )

            # Track if vehicle was ever above the early merge line
            early_line_y_now = line_y_at_x(center_x, dark_green_line)

            inside_early_region = (
                center_inside_early_region
                or bottom_inside_early_region
            )

            if (
                inside_early_region
                and center_y < early_line_y_now
            ):
                state["has_been_above_dark_green_line"] = True



            # EARLY MERGE FALLBACK:
            # If the car is inside the early region and has passed
            # the dark green early-merge line, count it as EARLY.
            # This catches cars that were already past the line
            # when tracking started.

            early_line_y_now = line_y_at_x(center_x, dark_green_line)

            inside_early_region = (
                center_inside_early_region
                or bottom_inside_early_region
            )

            past_early_line = (
                center_y >= early_line_y_now
            )

            if (
            state["has_entered_merge_corridor"]
            and inside_early_region
            and state["has_been_above_dark_green_line"]
            and past_early_line
            and not state.get("early_merge_latched", False)
            and not state.get("late_merge_latched", False)
        ):

                state["early_line_frame_count"] += 1

            else:

                state["early_line_frame_count"] = 0


            if (
                state["early_line_frame_count"] >= 3
                and not state.get("early_merge_latched", False)
                and not state.get("late_merge_latched", False)
            ):

                state["merge_type"] = "early"
                state["early_merge_latched"] = True

                early_merge_ids.add(track_id)
                early_merge_count = len(early_merge_ids)

                update_saved_merge_type(
                    traffic_behavior_rows,
                    track_id,
                    state["merge_type"]
                )


            state["prev_center_x"] = center_x
            state["prev_center_y"] = center_y


            # LATE MERGE:
            # Vehicle must come from the top merge lane and pass into yellow late region.

            dark_line_y_for_car = line_y_at_x(center_x, dark_green_line)

            if (
                center_inside_early_region
                and center_y < dark_line_y_for_car
            ):
                state["was_in_early_region"] = True

            now_left_of_blue = center_x <= late_blue_line[0][0]

            in_yellow_late_area = (
                center_inside_late_region
                and bottom_inside_late_region
            )

            crossed_into_late_without_early = (
                state["has_entered_merge_corridor"]
                and state["was_in_early_region"]
                and not state.get("early_merge_latched", False)
                and not state.get("late_merge_latched", False)
                and now_left_of_blue
                and in_yellow_late_area
            )

            if crossed_into_late_without_early:

              state["merge_type"] = "late"
              state["late_merge_latched"] = True

              late_merge_ids.add(track_id)
              late_merge_count = len(late_merge_ids)

              update_saved_merge_type(
                  traffic_behavior_rows,
                  track_id,
                  state["merge_type"]
              )



            # VEHICLE SIZE CLASSIFICATION WITH SMOOTHING

            state["vehicle_label"] = best_label

            raw_size_label = classify_vehicle_size(
                best_label,
                x1_full,
                y1_full,
                x2_full,
                y2_full,
                width
            )

            state["size_vote_history"].append(raw_size_label)

            if len(state["size_vote_history"]) > 8:
                state["size_vote_history"].pop(0)

            big_votes = state["size_vote_history"].count("big car")
            small_votes = state["size_vote_history"].count("small car")

            if big_votes >= 6:
                state["stable_vehicle_size"] = "big car"

            elif small_votes >= 6:
                state["stable_vehicle_size"] = "small car"

            state["vehicle_size"] = state["stable_vehicle_size"]

            vehicle_size = max(
                x2_full - x1_full,
                y2_full - y1_full
            )


            # STORE VEHICLE PIXEL SIZE DATA FOR HISTOGRAM

            box_width = x2_full - x1_full
            box_height = y2_full - y1_full
            box_area = box_width * box_height
            raw_vehicle_size = max(box_width, box_height)
            size_threshold = width * SIZE_THRESHOLD_SCALE

            vehicle_size_rows.append({
                "Frame": frame_count,
                "Track ID": track_id,
                "YOLO Label": best_label,
                "Box Width px": round(box_width, 2),
                "Box Height px": round(box_height, 2),
                "Box Area px": round(box_area, 2),
                "Max Dimension px": round(raw_vehicle_size, 2),
                "Threshold px": round(size_threshold, 2),
                "Classified Size": "BIG" if state["vehicle_size"] == "big car" else "SMALL"
            })


            # SPEED ESTIMATION: PIXELS/SECOND

            if state["prev_x"] is not None:

                dx = bottom_center_x - state["prev_x"]
                dy = bottom_center_y - state["prev_y"]

                pixel_distance = (dx**2 + dy**2) ** 0.5

                frames_elapsed = frame_count - state["prev_frame"]

                if frames_elapsed > 0:
                    speed_pixels_per_second = pixel_distance / (frames_elapsed / fps)
                else:
                    speed_pixels_per_second = 0

            else:
                speed_pixels_per_second = 0

            state["prev_x"] = bottom_center_x
            state["prev_y"] = bottom_center_y
            state["prev_frame"] = frame_count


            # SPEED ESTIMATION: MPH
            # Uses position history + feet-per-pixel calibration

            if "pos_history" not in state:
                state["pos_history"] = []

            state["pos_history"].append((bottom_center_x, bottom_center_y, frame_count))

            if len(state["pos_history"]) > SPEED_WINDOW:
                state["pos_history"].pop(0)

            prev_speed_mph = state["speed_mph"]

            if len(state["pos_history"]) >= 10:

                first_x, first_y, first_frame = state["pos_history"][0]
                last_x, last_y, last_frame = state["pos_history"][-1]

                dx = last_x - first_x
                dy = last_y - first_y

                pixel_distance = (dx**2 + dy**2) ** 0.5
                frames_elapsed = last_frame - first_frame

                if frames_elapsed > 0 and pixel_distance > MIN_PIXELS_MOVED:

                    time_elapsed = frames_elapsed / fps

                    pixels_per_second = pixel_distance / time_elapsed

                    feet_per_second = pixels_per_second * FEET_PER_PIXEL

                    speed_mph_raw = feet_per_second / 1.46667

                    speed_mph_raw = min(speed_mph_raw, MAX_REASONABLE_MPH)

                    speed_mph = MPH_SMOOTHING * prev_speed_mph + (1 - MPH_SMOOTHING) * speed_mph_raw

                else:
                    speed_mph = MPH_SMOOTHING * prev_speed_mph

            else:
                speed_mph = 0.0

            state["speed_mph"] = speed_mph


            # MERGE LOGIC
            touches_entry = box_touches_polygon(
                x1_full,
                y1_full,
                x2_full,
                y2_full,
                entry_polygon
            )

            touches_merge = box_touches_polygon(
                x1_full,
                y1_full,
                x2_full,
                y2_full,
                merge_polygon
            )

            bottom_limit_y = bottom_boundary_y_at_x(
                bottom_center_x,
                merge_polygon
            )

            left_merge_corridor_downward = bottom_center_y > bottom_limit_y + 8

            if not state["merge_done"]:

                if not state["is_merging"] and (touches_entry or touches_merge):

                    state["is_merging"] = True

                    state["merge_frame_count"] = 0

                    state["entry_frame"] = frame_count
                    state["entry_time"] = frame_count / fps

                # Stop early merge timing once the vehicle leaves the early green region.
                early_merge_finished = (
                    state["merge_type"] == "early"
                    and not inside_early_region
                )

                # Stop late merge timing once the vehicle leaves the yellow late region.
                late_merge_finished = (
                    state["merge_type"] == "late"
                    and not in_yellow_late_area
                )

                if state["is_merging"] and bottom_center_x <= exit_x:

                    state["is_merging"] = False
                    state["merge_done"] = True

                elif state["is_merging"] and left_merge_corridor_downward:

                    state["is_merging"] = False
                    state["merge_done"] = True

                elif state["is_merging"] and early_merge_finished:

                    state["is_merging"] = False
                    state["merge_done"] = True

                elif state["is_merging"] and late_merge_finished:

                    state["is_merging"] = False
                    state["merge_done"] = True

            merging_now = state["is_merging"]

            state["merge_frame_count"] = (
                state["merge_frame_count"] + 1
                if merging_now else 0
            )


            # STORE MERGE SPEEDS

            if merging_now:

                state["speed_values"].append(speed_pixels_per_second)
                state["mph_values"].append(speed_mph)


            # COUNT MERGES AND STORE TRAFFIC CSV ROW

            if merging_now and state["merge_frame_count"] >= min_merge_frames_to_count:

                if track_id not in counted_merge_ids:

                    duplicate_physical_car = False

                    for old_x, old_y, old_frame in recent_counted_centers:

                        dist = (
                            (bottom_center_x - old_x)**2 +
                            (bottom_center_y - old_y)**2
                        )**0.5

                        if dist < count_distance_thresh and frame_count - old_frame < count_memory_frames:

                            duplicate_physical_car = True
                            break

                    if not duplicate_physical_car:

                        counted_merge_ids.add(track_id)

                        recent_counted_centers.append(
                            (bottom_center_x, bottom_center_y, frame_count)
                        )

                        merge_count += 1

                        avg_speed_pixels = np.mean(state["speed_values"]) if len(state["speed_values"]) > 0 else 0
                        avg_speed_mph = np.mean(state["mph_values"]) if len(state["mph_values"]) > 0 else 0

                        traffic_behavior_rows.append({

                        "Merge ID": merge_count,

                        "Track ID": track_id,

                        "Merge Type": (
                            "EARLY"
                            if state["merge_type"] == "early"
                            else "LATE"
                            if state["merge_type"] == "late"
                            else "MERGING"
                        ),

                        "Vehicle Size": (
                            "BIG"
                            if state["vehicle_size"] == "big car"
                            else "SMALL"
                        ),

                        "Avg MPH": round(avg_speed_mph, 1),

                        "Merge Duration (s)": round(
                            (frame_count / fps) - state["entry_time"],
                            2
                        )
                    })


            # DRAW BOXES

            show_merge_label = not state["merge_done"]

            if show_merge_label and state["merge_type"] == "early":

                color = early_color

            elif show_merge_label and state["merge_type"] == "late":

                color = late_color

            elif show_merge_label and merging_now:

                color = currently_merging_color

            else:

                color = colors[region_idx]

            cv2.rectangle(
                annotated_full,
                (x1_full, y1_full),
                (x2_full, y2_full),
                color,
                2
            )


            # CLEAN LABEL DISPLAY

            short_type = "BIG" if state["vehicle_size"] == "big car" else "SMALL"
            mph_text = f"{int(round(speed_mph))} mph"

            label_text = f"{short_type} | {mph_text}"

            if show_merge_label and state["merge_type"] == "early":

                label_text = f"{short_type} | {mph_text} | EARLY MERGE"

            elif show_merge_label and state["merge_type"] == "late":

                label_text = f"{short_type} | {mph_text} | LATE MERGE"

            elif show_merge_label and merging_now:

                label_text = f"{short_type} | {mph_text} | MERGING"

            text_x = x1_full
            text_y = max(15, y1_full - 6)

            # White outline/shadow for readability
            cv2.putText(
                annotated_full,
                label_text,
                (text_x, text_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.36,
                (255, 255, 255),
                2
            )

            # Colored label on top
            cv2.putText(
                annotated_full,
                label_text,
                (text_x, text_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.36,
                color,
                1
            )


    # DRAW MERGE GUIDES + COUNT BOX

    overlay = annotated_full.copy()

    cv2.polylines(
        overlay,
        [merge_polygon.reshape((-1, 1, 2))],
        isClosed=True,
        color=(0, 0, 255),
        thickness=guide_thickness
    )

    cv2.polylines(
        overlay,
        [early_region_polygon.reshape((-1, 1, 2))],
        isClosed=True,
        color=(0, 255, 0),
        thickness=guide_thickness
    )

    cv2.polylines(
        overlay,
        [late_region_polygon.reshape((-1, 1, 2))],
        isClosed=True,
        color=(0, 255, 255),
        thickness=guide_thickness
    )

    cv2.line(
        overlay,
        tuple(late_blue_line[0]),
        tuple(late_blue_line[1]),
        (255, 0, 0),
        guide_thickness + 2
    )

    cv2.line(
        overlay,
        tuple(dark_green_line[0]),
        tuple(dark_green_line[1]),
        (0, 120, 0),
        guide_thickness + 3
    )

    annotated_full = cv2.addWeighted(
        overlay,
        guide_alpha,
        annotated_full,
        1 - guide_alpha,
        0
    )

    cv2.rectangle(
        annotated_full,
        (25, 420),
        (470, 505),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        annotated_full,
        f"Early Tracking: {early_merge_count}",
        (40, 455),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 0),
        2
    )

    cv2.putText(
        annotated_full,
        f"Late Tracking: {late_merge_count}",
        (40, 490),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 255),
        2
    )

    out.write(annotated_full)

    frame_count += 1


# RELEASE VIDEO

cap.release()
out.release()


# SAVE CLEAN TRAFFIC CSV

simple_csv = "traffic_behavior_summary.csv"

with open(simple_csv, "w", newline="") as file:

    writer = csv.DictWriter(file, fieldnames=[

    "Merge ID",
    "Track ID",
    "Merge Type",
    "Vehicle Size",
    "Avg MPH",
    "Merge Duration (s)"
])

    writer.writeheader()
    writer.writerows(traffic_behavior_rows)

print("Saved:", simple_csv)

files.download(simple_csv)


# OVERALL VEHICLE SIZE STATISTICS CSV

big_rows = [
    r for r in traffic_behavior_rows
    if r["Vehicle Size"] == "BIG"
]

small_rows = [
    r for r in traffic_behavior_rows
    if r["Vehicle Size"] == "SMALL"
]

stats_rows = []

for label, rows in [("BIG", big_rows), ("SMALL", small_rows)]:

    if len(rows) == 0:
        continue

    avg_mph = np.mean([r["Avg MPH"] for r in rows])

    avg_duration = np.mean([
        r["Merge Duration (s)"]
        for r in rows
    ])

    stats_rows.append({

        "Vehicle Size": label,

        "Merge Count": len(rows),

        "Avg MPH": round(avg_mph, 1),

        "Avg Merge Duration (s)": round(avg_duration, 2)
    })

stats_csv = "vehicle_statistics_summary.csv"

with open(stats_csv, "w", newline="") as file:

    writer = csv.DictWriter(file, fieldnames=[

        "Vehicle Size",
        "Merge Count",
        "Avg MPH",
        "Avg Merge Duration (s)"
    ])

    writer.writeheader()
    writer.writerows(stats_rows)

print("Saved:", stats_csv)

files.download(stats_csv)


# EARLY VS LATE MERGE STATISTICS CSV

early_rows = [
    r for r in traffic_behavior_rows
    if r["Merge Type"] == "EARLY"
]

late_rows = [
    r for r in traffic_behavior_rows
    if r["Merge Type"] == "LATE"
]

merge_type_stats_rows = []

for label, rows in [("EARLY", early_rows), ("LATE", late_rows)]:

    if len(rows) == 0:
        continue

    avg_mph = np.mean([r["Avg MPH"] for r in rows])

    avg_duration = np.mean([
        r["Merge Duration (s)"]
        for r in rows
    ])

    merge_type_stats_rows.append({

        "Merge Type": label,

        "Merge Count": len(rows),

        "Avg MPH": round(avg_mph, 1),

        "Avg Merge Duration (s)": round(avg_duration, 2)
    })

merge_type_stats_csv = "early_late_merge_statistics_summary.csv"

with open(merge_type_stats_csv, "w", newline="") as file:

    writer = csv.DictWriter(file, fieldnames=[

        "Merge Type",
        "Merge Count",
        "Avg MPH",
        "Avg Merge Duration (s)"
    ])

    writer.writeheader()
    writer.writerows(merge_type_stats_rows)

print("Saved:", merge_type_stats_csv)

files.download(merge_type_stats_csv)

# EARLY VS LATE AVG MPH BAR GRAPH

if len(merge_type_stats_rows) > 0:

    merge_types = [
        row["Merge Type"]
        for row in merge_type_stats_rows
    ]

    avg_mph_values = [
        row["Avg MPH"]
        for row in merge_type_stats_rows
    ]

    plt.figure(figsize=(7, 5))

    plt.bar(
        merge_types,
        avg_mph_values
    )

    plt.xlabel("Merge Type")
    plt.ylabel("Average MPH During Merge")
    plt.title("Average MPH of Early vs Late Merging Vehicles")

    plt.tight_layout()

    early_late_graph_path = "early_vs_late_avg_mph.png"

    plt.savefig(early_late_graph_path, dpi=300)

    plt.close()

    print("Saved:", early_late_graph_path)

    files.download(early_late_graph_path)


# EARLY/LATE BY VEHICLE SIZE STATISTICS CSV

merge_size_stats_rows = []

groups = [
    ("EARLY", "BIG"),
    ("EARLY", "SMALL"),
    ("LATE", "BIG"),
    ("LATE", "SMALL")
]

for merge_type, vehicle_size in groups:

    rows = [
        r for r in traffic_behavior_rows
        if r["Merge Type"] == merge_type
        and r["Vehicle Size"] == vehicle_size
    ]

    if len(rows) == 0:
        avg_mph = 0
        avg_duration = 0
    else:
        avg_mph = np.mean([r["Avg MPH"] for r in rows])
        avg_duration = np.mean([r["Merge Duration (s)"] for r in rows])

    merge_size_stats_rows.append({
        "Group": f"{merge_type} {vehicle_size}",
        "Merge Type": merge_type,
        "Vehicle Size": vehicle_size,
        "Merge Count": len(rows),
        "Avg MPH": round(avg_mph, 1),
        "Avg Merge Duration (s)": round(avg_duration, 2)
    })

merge_size_stats_csv = "early_late_by_vehicle_size_summary.csv"

with open(merge_size_stats_csv, "w", newline="") as file:

    writer = csv.DictWriter(file, fieldnames=[
        "Group",
        "Merge Type",
        "Vehicle Size",
        "Merge Count",
        "Avg MPH",
        "Avg Merge Duration (s)"
    ])

    writer.writeheader()
    writer.writerows(merge_size_stats_rows)

print("Saved:", merge_size_stats_csv)
files.download(merge_size_stats_csv)


# AVG MPH BY MERGE TYPE AND VEHICLE SIZE GRAPH

group_labels = [
    f'{row["Group"]}\n(n={row["Merge Count"]})'
    for row in merge_size_stats_rows
]

avg_mph_values = [
    row["Avg MPH"]
    for row in merge_size_stats_rows
]

plt.figure(figsize=(9, 5))
plt.bar(group_labels, avg_mph_values)

plt.xlabel("Merge Type and Vehicle Size")
plt.ylabel("Average MPH During Merge")
plt.title("Average MPH by Merge Type and Vehicle Size")

for i, value in enumerate(avg_mph_values):
    plt.text(
        i,
        value + 0.3,
        f"{value} mph",
        ha="center"
    )

plt.tight_layout()

merge_size_mph_graph_path = "early_late_avg_mph_by_vehicle_size.png"

plt.savefig(merge_size_mph_graph_path, dpi=300)
plt.close()

print("Saved:", merge_size_mph_graph_path)
files.download(merge_size_mph_graph_path)


# AVG MERGE DURATION BY MERGE TYPE AND VEHICLE SIZE GRAPH

avg_duration_values = [
    row["Avg Merge Duration (s)"]
    for row in merge_size_stats_rows
]

plt.figure(figsize=(9, 5))
plt.bar(group_labels, avg_duration_values)

plt.xlabel("Merge Type and Vehicle Size")
plt.ylabel("Average Merge Duration (s)")
plt.title("Average Merge Duration by Merge Type and Vehicle Size")

for i, value in enumerate(avg_duration_values):
    plt.text(
        i,
        value + 0.05,
        f"{value}s",
        ha="center"
    )

plt.tight_layout()

merge_size_duration_graph_path = "early_late_duration_by_vehicle_size.png"

plt.savefig(merge_size_duration_graph_path, dpi=300)
plt.close()

print("Saved:", merge_size_duration_graph_path)
files.download(merge_size_duration_graph_path)


# SAVE VEHICLE PIXEL SIZE DATA

vehicle_size_csv = "vehicle_pixel_size_data.csv"

with open(vehicle_size_csv, "w", newline="") as file:

    writer = csv.DictWriter(file, fieldnames=[

        "Frame",
        "Track ID",
        "YOLO Label",
        "Box Width px",
        "Box Height px",
        "Box Area px",
        "Max Dimension px",
        "Threshold px",
        "Classified Size"
    ])

    writer.writeheader()
    writer.writerows(vehicle_size_rows)

print("Saved:", vehicle_size_csv)

files.download(vehicle_size_csv)

# GRAPH VEHICLE PIXEL SIZE DISTRIBUTION

max_sizes = [
    r["Max Dimension px"]
    for r in vehicle_size_rows
]

plt.figure(figsize=(10, 6))

plt.hist(max_sizes, bins=30)

plt.axvline(
    width * SIZE_THRESHOLD_SCALE,
    linestyle="--",
    label="Current threshold"
)

plt.xlabel("Vehicle max bounding-box dimension (pixels)")
plt.ylabel("Number of detections")
plt.title("Vehicle Pixel Size Distribution")
plt.legend()
plt.grid(True)

histogram_path = "vehicle_pixel_size_histogram.png"

plt.savefig(histogram_path, dpi=300)

plt.close()

print("Saved:", histogram_path)

files.download(histogram_path)


# DONE

print("Done!")
print("Saved as:", output_path)

files.download(output_path)

Upload your trained best.pt file


Saving best (6).pt to best (6).pt
Upload your traffic video file


Saving TruckAnalysis1 (2).mp4 to TruckAnalysis1 (2).mp4
FPS: 30.0
Frame size: 1920 x 1080
Total frames: 1231
Vehicle size threshold scale: 0.0375
Vehicle size threshold px: 72.0
Saved: traffic_behavior_summary.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!
Saved as: early_late_merge_identifier_with_csvs_histogram_and_mph.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>